# EvolveAI — T4 GPU Autonomous Learning Starter

This notebook runs exactly **one bounded SNN training job** on a CUDA GPU using the allowlisted TinyStories dataset. It saves a separate checkpoint and report to Google Drive without launching repeated trials or modifying the historical governor champion.

**Important:** this is the first controlled prototype, not a conscious or superintelligent system. The current experiment selector searches reviewed configurations; it does not yet invent arbitrary learning rules. Internet access is restricted to the approved Hugging Face dataset path and the notebook does not execute downloaded code.

In [ ]:
from pathlib import Path
import subprocess, os, sys

REPO_DIR = Path('/content/EvolveAI')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/swasthik-s/EvolveAI.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/EvolveAI/snn
%pip -q install 'datasets>=3.0,<5.0'
print('Repository ready:', REPO_DIR)


In [ ]:
import torch
from train import resolve_device

device = resolve_device('auto')
print('PyTorch:', torch.__version__)
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    raise RuntimeError('CUDA GPU not detected. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')


## 1. Connect persistent storage

Colab runtimes can reset. Mount Drive so the governor history and model checkpoint survive disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/EvolveAI')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent experiment folder:', RUN_DIR)


## 2. Acquire approved learning data

This streams at most 5 MiB from the allowlisted TinyStories dataset, deduplicates records, and creates a provenance manifest. Review the upstream dataset card and license before any use beyond research.

In [ ]:
import json
from dataset_harvester import fetch_corpus

CORPUS_PATH = RUN_DIR / 'tiny_stories_sample.txt'
MANIFEST_PATH = RUN_DIR / 'tiny_stories_manifest.json'

# Reuse the exact corpus from the previous cycle so the hash stays stable
# and compatible checkpoints can be continued safely.
if CORPUS_PATH.exists() and MANIFEST_PATH.exists():
    corpus = CORPUS_PATH.read_text(encoding='utf-8')
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    print('Reusing saved corpus from Drive.')
else:
    corpus, manifest = fetch_corpus('TinyStories (research starter)', max_mb=5.0)
    CORPUS_PATH.write_text(corpus, encoding='utf-8')
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Corpus bytes:', len(corpus.encode('utf-8')))
print('Records kept:', manifest['records_kept'])
print('SHA-256:', manifest['corpus_sha256'])
print('Dataset:', manifest['dataset_url'])
print('License note:', manifest['commercial_use_note'])


## 3. Run one-shot training

This launches exactly one fresh model training job. It does not run a multi-trial sweep or repeated autonomous cycles. The result is saved separately in Drive; the previous governor history and champion checkpoint are preserved.

GPU quotas are enforced by Colab. A single invocation avoids repeated launches, but does not bypass the account's GPU usage limits.

In [ ]:
# One invocation: one model, one training job, one saved report.
import importlib, subprocess, sys
subprocess.run(['git', '-C', '/content/EvolveAI', 'pull', '--ff-only'], check=True)

# Refresh modules in dependency order.
import data
importlib.reload(data)
import train
importlib.reload(train)
import governor
importlib.reload(governor)
import one_shot
importlib.reload(one_shot)
from one_shot import run_one_shot

report = run_one_shot(
    corpus=corpus,
    manifest=manifest,
    output_dir=RUN_DIR,
    steps=600,
    block_size=32,
    batch_size=8,
    hidden_size=64,
    learning_rate=0.002,
    seed=42,
    architecture='baseline',
    device='cuda',
)

print('Mode:', report['mode'])
print('Device:', report['device'])
print('Steps completed:', report['steps_completed'])
print('Final training loss:', report['final_train_loss'])
print('Final validation loss:', report['final_validation_loss'])
print('Validation perplexity:', report['final_validation_perplexity'])
print('Elapsed seconds:', report['elapsed_seconds'])
print('Checkpoint:', report['checkpoint_path'])
print('Report:', RUN_DIR / 'one_shot_report.json')


In [ ]:
import json
report = json.loads((RUN_DIR / 'one_shot_report.json').read_text(encoding='utf-8'))
print('One-shot training report')
print('Steps:', report['steps_completed'])
print('Validation loss:', report['final_validation_loss'])
print('Checkpoint:', report['checkpoint_path'])

# Historical governed cycles remain unchanged.
state_path = RUN_DIR / 'governor_state.json'
if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print('Previous governed cycles preserved:', len(state.get('cycles', [])))


## What this first version does—and does not do

- **Does:** train one model in one invocation, save a separate checkpoint and report, and preserve prior governor history.
- **Does not yet:** keep a Colab GPU process alive indefinitely, autonomously browse arbitrary internet pages, invent unrestricted code, or automatically replace the model serving the frontend.
- **Next milestone:** evaluate the frozen one-shot checkpoint on a genuinely independent corpus before considering promotion.

A Colab notebook is a development worker, not a reliable always-on API server. Do not expose the notebook runtime publicly with an unauthenticated tunnel.